# Extraction de features

> Récupération des features internes, logits, reconstructions et latents.


In [ ]:
#| default_exp features


In [ ]:
#| export
"""Feature extraction from the Human-Centered AAE internals."""


import torch
from torch import Tensor, nn

from tell_me_why.config import HumanCenteredFeatures


@torch.no_grad()
def extract_human_aae_features(
    aae: nn.Module,
    inputs: Tensor,
    *,
    device: torch.device | str | None = None,
) -> HumanCenteredFeatures:
    """Return latent vectors, encoder feature maps, logits, and reconstruction."""

    was_training = aae.training
    aae.eval()
    if device is not None:
        aae.to(device)
        inputs = inputs.to(device)

    encoder_features = aae.unet.layers[0](inputs)
    logits = aae(inputs)
    latents = aae.zi
    reconstruction = getattr(aae, "decoder_output", None)
    aae.train(was_training)

    return HumanCenteredFeatures(
        logits=logits.detach().cpu(),
        latents=latents.detach().cpu(),
        encoder_features=encoder_features.detach().cpu(),
        reconstruction=None if reconstruction is None else reconstruction.detach().cpu(),
    )
